## Find SWOT Pass Numbers for a New Region

Use the cells below to discover which pass numbers cover any ocean area.  
Steps:
1. Run the **Download** cell once to get the AVISO SWOT swath shapefile.
2. Edit the bounding box in the **Find Passes** cell and run it.

In [6]:
# ============================================================
# STEP 1 — Download SWOT orbit swath shapefile (run once)
# ============================================================
import os, zipfile, urllib.request

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "swot_orbit_data")
os.makedirs(DATA_DIR, exist_ok=True)

SWATH_ZIP = os.path.join(DATA_DIR, "sph_science_swath.zip")
URL = "https://www.aviso.altimetry.fr/fileadmin/documents/missions/Swot/sph_science_swath.zip"

if not os.path.exists(SWATH_ZIP):
    print("Downloading SWOT swath shapefile (~10 MB)...")
    urllib.request.urlretrieve(URL, SWATH_ZIP)
    print(f"Saved to: {SWATH_ZIP}")
else:
    print(f"Already downloaded: {SWATH_ZIP}")

print("Done. Proceed to the next cell.")

Already downloaded: c:\Users\abhik\Desktop\project related work\swot_orbit_data\sph_science_swath.zip
Done. Proceed to the next cell.


In [7]:
# ============================================================
# STEP 2 — Find pass numbers for YOUR new region
# ============================================================
import geopandas as gpd
from shapely.geometry import box

# -------  EDIT THIS BOUNDING BOX  -------
# Format: (min_lon, min_lat, max_lon, max_lat)
# Example regions (uncomment the one you want, or set your own):

# Bay of Bengal (original area)
BBOX = (80.0, 5.0, 100.0, 25.0)

# Arabian Sea
# BBOX = (50.0, 5.0, 78.0, 28.0)

# South China Sea
# BBOX = (105.0, 0.0, 122.0, 22.0)

# ← SET YOUR OWN REGION HERE ↓
# BBOX = (50.0, 5.0, 78.0, 28.0)   # <-- change these coordinates
# -----------------------------------------

swath_zip = os.path.join(DATA_DIR, "sph_science_swath.zip")
print(f"Loading shapefile from: {swath_zip}")
swath = gpd.read_file(swath_zip)

# Make sure CRS matches
region = box(*BBOX)
swath_wgs84 = swath.to_crs(epsg=4326)

# Find intersecting passes
mask = swath_wgs84.intersects(region)
matching = swath_wgs84[mask]

# Extract pass numbers (3-digit zero-padded)
pass_col = None
for col in ["ID_PASS", "pass", "PASS", "Pass"]:
    if col in matching.columns:
        pass_col = col
        break

if pass_col is None:
    print("Columns available:", list(matching.columns))
    raise ValueError("Could not find pass-number column — check the column name above")

pass_numbers = sorted({str(int(p)).zfill(3) for p in matching[pass_col].dropna()})

print(f"\nRegion: lon [{BBOX[0]}, {BBOX[2]}]  lat [{BBOX[1]}, {BBOX[3]}]")
print(f"Found {len(pass_numbers)} passes:\n")
print(pass_numbers)
print("\n--- Copy-paste ready for DESIRED_PASSES ---")
print("DESIRED_PASSES = {")
chunk = 5
for i in range(0, len(pass_numbers), chunk):
    row = ", ".join(f'"{p}"' for p in pass_numbers[i:i+chunk])
    print(f'    {row},')
print("}")

Loading shapefile from: c:\Users\abhik\Desktop\project related work\swot_orbit_data\sph_science_swath.zip

Region: lon [80.0, 100.0]  lat [5.0, 25.0]
Found 40 passes:

['008', '023', '036', '049', '064', '077', '092', '105', '133', '146', '161', '174', '189', '202', '217', '230', '245', '258', '273', '286', '301', '314', '342', '355', '370', '383', '398', '411', '424', '439', '452', '467', '480', '495', '508', '523', '536', '551', '564', '579']

--- Copy-paste ready for DESIRED_PASSES ---
DESIRED_PASSES = {
    "008", "023", "036", "049", "064",
    "077", "092", "105", "133", "146",
    "161", "174", "189", "202", "217",
    "230", "245", "258", "273", "286",
    "301", "314", "342", "355", "370",
    "383", "398", "411", "424", "439",
    "452", "467", "480", "495", "508",
    "523", "536", "551", "564", "579",
}


# SWOT File Sorter — Filter by Pass Number  \n\nThis script:\n1. Reads a source folder containing cycle subfolders (e.g., `cycle_001`, `cycle_002`, etc.)\n2. Filters `.nc` files based on hardcoded pass numbers\n3. Copies matching files to a new output directory preserving the cycle folder structure

In [1]:
import os
import shutil
import re
from datetime import datetime

# ============================================================
# CONFIGURATION — Edit these paths, pass numbers, and dates
# ============================================================

# Source folder (can have any depth of subfolders — cycle folders,
# forward/reproc groupings, etc.)
SOURCE_DIR = r"Y:\level 3 ocean basic files\Basic\reproc"

# Output folder where filtered files will be copied
OUTPUT_DIR = r"Y:\level 3 ocean basic files\Basic_filters_bob"

# ---------- HARDCODED PASS NUMBERS YOU WANT TO KEEP ----------
DESIRED_PASSES = {
    "008", "023", "036", "049", "064",
    "077", "092", "105", "133", "146",
    "161", "174", "189", "202", "217",
    "230", "245", "258", "273", "286",
    "301", "314", "342", "355", "370",
    "383", "398", "411", "424", "439",
    "452", "467", "480", "495", "508",
    "523", "536", "551", "564", "579",
}

# ---------- HARDCODED DATE RANGE (set None to disable) -------
# Format: "YYYYMMDD"
# START_DATE = "20230401"   # include files ON or AFTER this date
# END_DATE   = "20240331"   # include files ON or BEFORE this date
# Set both to None to skip date filtering:
START_DATE = None
END_DATE   = None
# ============================================================


def extract_pass_number(filename):
    """
    Extract the 3-digit pass number from a SWOT filename.
    Pattern: ..._CCC_PPP_YYYYMMDDTHHMMSS...  →  returns PPP string
    """
    match = re.search(r'_(\d{3})_(\d{3})_\d{8}T', filename)
    if match:
        return match.group(2)
    return None


def extract_date(filename):
    """
    Extract the date from a SWOT filename.
    Pattern: ..._CCC_PPP_YYYYMMDDTHHMMSS...  →  returns datetime object
    Returns None if the date cannot be parsed.
    """
    match = re.search(r'_\d{3}_\d{3}_(\d{8})T', filename)
    if match:
        try:
            return datetime.strptime(match.group(1), "%Y%m%d")
        except ValueError:
            return None
    return None


def is_in_date_range(filename, start_date_str, end_date_str):
    """
    Returns True if the file's embedded date falls within [start_date, end_date].
    If start_date_str or end_date_str is None, that bound is ignored.
    Returns True if the date cannot be extracted (fail-open).
    """
    file_date = extract_date(filename)
    if file_date is None:
        return True  # can't determine date → don't exclude

    if start_date_str:
        start = datetime.strptime(start_date_str, "%Y%m%d")
        if file_date < start:
            return False

    if end_date_str:
        end = datetime.strptime(end_date_str, "%Y%m%d")
        if file_date > end:
            return False

    return True


def sort_and_copy_files(source_dir, output_dir, desired_passes,
                        start_date=None, end_date=None):
    """
    Recursively walk through source_dir.
    For every .nc file whose pass number is in desired_passes AND whose
    date falls within [start_date, end_date], copy it to output_dir
    preserving the FULL relative folder structure.
    """
    total_copied = 0
    total_skipped = 0
    folder_summary = {}   # relative_folder → {copied, skipped, files}

    for root, dirs, files in os.walk(source_dir):
        rel_path = os.path.relpath(root, source_dir)

        nc_files = [f for f in files if f.lower().endswith('.nc')]
        if not nc_files:
            continue

        copied_files = []
        skipped = 0

        for filename in sorted(nc_files):
            pass_num = extract_pass_number(filename)
            in_range = is_in_date_range(filename, start_date, end_date)

            if pass_num and pass_num in desired_passes and in_range:
                dest_folder = os.path.join(output_dir, rel_path)
                os.makedirs(dest_folder, exist_ok=True)

                src_path = os.path.join(root, filename)
                dst_path = os.path.join(dest_folder, filename)
                shutil.copy2(src_path, dst_path)
                copied_files.append(filename)
                total_copied += 1
            else:
                skipped += 1
                total_skipped += 1

        folder_summary[rel_path] = {
            'copied': len(copied_files),
            'skipped': skipped,
            'files': copied_files
        }

    # ---- Print Summary ----
    date_range_str = (
        f"{start_date or 'any'} → {end_date or 'any'}"
    )
    print("=" * 70)
    print(f"  SORTING COMPLETE")
    print(f"  Source      : {source_dir}")
    print(f"  Output      : {output_dir}")
    print(f"  Passes      : {', '.join(sorted(desired_passes))}")
    print(f"  Date range  : {date_range_str}")
    print("=" * 70)
    print(f"\n  Total files copied  : {total_copied}")
    print(f"  Total files skipped : {total_skipped}")
    print(f"  Folders processed   : {len(folder_summary)}\n")

    for folder in sorted(folder_summary.keys()):
        info = folder_summary[folder]
        if info['copied'] > 0 or info['skipped'] > 0:
            print(f"  📁 {folder}/  ({info['copied']} copied, {info['skipped']} skipped)")
            for f in info['files']:
                print(f"      ✔ {f}")

    print("\n" + "=" * 70)
    return folder_summary


# ============================================================
# RUN
# ============================================================
print(f"Desired passes : {sorted(DESIRED_PASSES)}")
print(f"Date range     : {START_DATE or 'any'} → {END_DATE or 'any'}\n")
summary = sort_and_copy_files(SOURCE_DIR, OUTPUT_DIR, DESIRED_PASSES,
                              start_date=START_DATE, end_date=END_DATE)


Desired passes : ['008', '023', '036', '049', '064', '077', '092', '105', '133', '146', '161', '174', '189', '202', '217', '230', '245', '258', '273', '286', '301', '314', '342', '355', '370', '383', '398', '411', '424', '439', '452', '467', '480', '495', '508', '523', '536', '551', '564', '579']
Date range     : any → any

  SORTING COMPLETE
  Source      : Y:\level 3 ocean basic files\Basic\reproc
  Output      : Y:\level 3 ocean basic files\Basic_filters_bob
  Passes      : 008, 023, 036, 049, 064, 077, 092, 105, 133, 146, 161, 174, 189, 202, 217, 230, 245, 258, 273, 286, 301, 314, 342, 355, 370, 383, 398, 411, 424, 439, 452, 467, 480, 495, 508, 523, 536, 551, 564, 579
  Date range  : any → any

  Total files copied  : 1728
  Total files skipped : 23543
  Folders processed   : 45

  📁 cycle_001/  (29 copied, 380 skipped)
      ✔ SWOT_L3_LR_SSH_Basic_001_161_20230726T224518_20230726T233644_v3.0.nc
      ✔ SWOT_L3_LR_SSH_Basic_001_174_20230727T095407_20230727T104533_v3.0.nc
      ✔ SW

In [2]:
# ============================================================
# VERIFY — Quick count of files in output directory
# ============================================================

def verify_output(output_dir):
    """Print a compact summary of the output directory."""
    if not os.path.exists(output_dir):
        print("Output directory does not exist yet. Run the sorting cell first.")
        return
    
    total = 0
    folder_count = 0
    for root, dirs, files in os.walk(output_dir):
        nc_files = [f for f in files if f.lower().endswith('.nc')]
        if nc_files:
            rel = os.path.relpath(root, output_dir)
            print(f"  📁 {rel}: {len(nc_files)} files")
            total += len(nc_files)
            folder_count += 1
    
    print(f"\n{'='*50}")
    print(f"  Total folders: {folder_count}")
    print(f"  Total .nc files: {total}")

verify_output(OUTPUT_DIR)

  📁 cycle_001: 29 files
  📁 cycle_002: 40 files
  📁 cycle_003: 40 files
  📁 cycle_004: 30 files
  📁 cycle_005: 40 files
  📁 cycle_006: 40 files
  📁 cycle_007: 40 files
  📁 cycle_008: 30 files
  📁 cycle_009: 39 files
  📁 cycle_010: 40 files
  📁 cycle_011: 39 files
  📁 cycle_012: 39 files
  📁 cycle_013: 40 files
  📁 cycle_014: 39 files
  📁 cycle_015: 34 files
  📁 cycle_016: 39 files
  📁 cycle_017: 40 files
  📁 cycle_018: 40 files
  📁 cycle_019: 39 files
  📁 cycle_020: 40 files
  📁 cycle_021: 40 files
  📁 cycle_022: 38 files
  📁 cycle_023: 38 files
  📁 cycle_024: 39 files
  📁 cycle_025: 40 files
  📁 cycle_026: 36 files
  📁 cycle_027: 40 files
  📁 cycle_028: 40 files
  📁 cycle_029: 40 files
  📁 cycle_030: 40 files
  📁 cycle_031: 39 files
  📁 cycle_032: 40 files
  📁 cycle_033: 40 files
  📁 cycle_034: 39 files
  📁 cycle_035: 40 files
  📁 cycle_036: 39 files
  📁 cycle_037: 39 files
  📁 cycle_038: 38 files
  📁 cycle_039: 39 files
  📁 cycle_040: 40 files
  📁 cycle_041: 40 files
  📁 cycle_042: 3

## Cyclone-wise File Sorter

Filters and copies `.nc` files into separate folders for each cyclone and its three periods (before / cyclone / after).  
Reuses `SOURCE_DIR`, `DESIRED_PASSES`, and helper functions defined in the cell above.

In [5]:
# ============================================================
# CYCLONE ENTIRE-PERIOD SORTER
# One folder per cyclone (from earliest "before" start to latest "after" end)
# ============================================================

entire_period_totals = {}

print("=" * 70)
print("  CYCLONE ENTIRE-PERIOD SORTER")
print(f"  Source      : {OUTPUT_DIR}")
print(f"  Base output : {CYCLONE_OUTPUT_BASE}")
print("=" * 70)

for cyclone_name, periods in CYCLONES.items():
    # Find full window across before/cyclone/after
    start_dates = [d[0] for d in periods.values()]
    end_dates = [d[1] for d in periods.values()]
    overall_start = min(start_dates)   # YYYYMMDD works with min/max
    overall_end = max(end_dates)

    # Single folder for this cyclone
    out_dir = os.path.join(CYCLONE_OUTPUT_BASE, cyclone_name)

    result = sort_and_copy_files(
        OUTPUT_DIR,
        out_dir,
        DESIRED_PASSES,
        start_date=overall_start,
        end_date=overall_end
    )

    copied = sum(v["copied"] for v in result.values())
    skipped = sum(v["skipped"] for v in result.values())

    entire_period_totals[cyclone_name] = {
        "start": overall_start,
        "end": overall_end,
        "copied": copied,
        "skipped": skipped,
        "output": out_dir,
    }

    print(
        f"\n🌀 {cyclone_name}: {overall_start} → {overall_end}"
        f"\n   ✔ copied: {copied}   ✗ skipped: {skipped}"
        f"\n   📁 {out_dir}"
    )

print("\n" + "=" * 70)
print("  DONE — Entire-period cyclone folders created")
for cyclone_name, info in entire_period_totals.items():
    print(f"  {cyclone_name:10s}  {info['start']} → {info['end']}  |  {info['copied']} files")
print("=" * 70)

  CYCLONE ENTIRE-PERIOD SORTER
  Source      : Y:\level 3 ocean basic files\Basic_filters_bob
  Base output : Y:\level 3 ocean basic files\Basic_filters_cyclones
  SORTING COMPLETE
  Source      : Y:\level 3 ocean basic files\Basic_filters_bob
  Output      : Y:\level 3 ocean basic files\Basic_filters_cyclones\Shakti
  Passes      : 008, 023, 036, 049, 064, 077, 092, 105, 133, 146, 161, 174, 189, 202, 217, 230, 245, 258, 273, 286, 301, 314, 342, 355, 370, 383, 398, 411, 424, 439, 452, 467, 480, 495, 508, 523, 536, 551, 564, 579
  Date range  : 20240502 → 20240604

  Total files copied  : 59
  Total files skipped : 1669
  Folders processed   : 45

  📁 cycle_001/  (0 copied, 29 skipped)
  📁 cycle_002/  (0 copied, 40 skipped)
  📁 cycle_003/  (0 copied, 40 skipped)
  📁 cycle_004/  (0 copied, 30 skipped)
  📁 cycle_005/  (0 copied, 40 skipped)
  📁 cycle_006/  (0 copied, 40 skipped)
  📁 cycle_007/  (0 copied, 40 skipped)
  📁 cycle_008/  (0 copied, 30 skipped)
  📁 cycle_009/  (0 copied, 39 ski

In [6]:
# ============================================================
# CYCLONE CONFIGURATION
# ============================================================

# Root output folder — subfolders created automatically per cyclone / period
CYCLONE_OUTPUT_BASE = r"Y:\level 3 ocean basic files\Basic_filters_cyclones"

# Each cyclone → three periods with (start_date, end_date) in "YYYYMMDD"
CYCLONES = {
    "Shakti": {
        "before":  ("20240502", "20240516"),
        "cyclone": ("20240517", "20240520"),
        "after":   ("20240521", "20240604"),
    },
    "Remal": {
        "before":  ("20240509", "20240523"),
        "cyclone": ("20240524", "20240528"),
        "after":   ("20240529", "20240612"),
    },
    "Dana": {
        "before":  ("20241007", "20241021"),
        "cyclone": ("20241022", "20241026"),
        "after":   ("20241027", "20241110"),
    },
}

# ============================================================
# LOOP — one pass per cyclone × period
# Uses SOURCE_DIR, DESIRED_PASSES, sort_and_copy_files from cell above
# ============================================================
grand_copied  = 0
grand_skipped = 0

print("=" * 70)
print("  CYCLONE FILE SORTER")
print(f"  Source      : {OUTPUT_DIR}")
print(f"  Base output : {CYCLONE_OUTPUT_BASE}")
print("=" * 70)

for cyclone_name, periods in CYCLONES.items():
    print(f"\n  🌀 {cyclone_name}")
    print(f"  {'-' * 50}")

    for period_label, (start, end) in periods.items():
        # Output path: BASE / CycloneName / period_label
        out_dir = os.path.join(CYCLONE_OUTPUT_BASE, cyclone_name, period_label)

        result = sort_and_copy_files(
            OUTPUT_DIR, out_dir, DESIRED_PASSES,
            start_date=start, end_date=end
        )

        # sort_and_copy_files returns folder_summary dict
        copied  = sum(v["copied"]  for v in result.values())
        skipped = sum(v["skipped"] for v in result.values())
        grand_copied  += copied
        grand_skipped += skipped

        start_fmt = datetime.strptime(start, "%Y%m%d").strftime("%d %b %Y")
        end_fmt   = datetime.strptime(end,   "%Y%m%d").strftime("%d %b %Y")
        print(f"    [{period_label:8s}]  {start_fmt} → {end_fmt}"
              f"   ✔ {copied} copied   ✗ {skipped} skipped")

print("\n" + "=" * 70)
print(f"  ALL DONE  |  Total copied: {grand_copied}  |  Total skipped: {grand_skipped}")
print("=" * 70)


  CYCLONE FILE SORTER
  Source      : Y:\level 3 ocean basic files\Basic_filters_bob
  Base output : Y:\level 3 ocean basic files\Basic_filters_cyclones

  🌀 Shakti
  --------------------------------------------------
  SORTING COMPLETE
  Source      : Y:\level 3 ocean basic files\Basic_filters_bob
  Output      : Y:\level 3 ocean basic files\Basic_filters_cyclones\Shakti\before
  Passes      : 008, 023, 036, 049, 064, 077, 092, 105, 133, 146, 161, 174, 189, 202, 217, 230, 245, 258, 273, 286, 301, 314, 342, 355, 370, 383, 398, 411, 424, 439, 452, 467, 480, 495, 508, 523, 536, 551, 564, 579
  Date range  : 20240502 → 20240516

  Total files copied  : 23
  Total files skipped : 1705
  Folders processed   : 45

  📁 cycle_001/  (0 copied, 29 skipped)
  📁 cycle_002/  (0 copied, 40 skipped)
  📁 cycle_003/  (0 copied, 40 skipped)
  📁 cycle_004/  (0 copied, 30 skipped)
  📁 cycle_005/  (0 copied, 40 skipped)
  📁 cycle_006/  (0 copied, 40 skipped)
  📁 cycle_007/  (0 copied, 40 skipped)
  📁 cycle

In [7]:
print("=" * 70)
print("  VERIFICATION — Cyclone output folder")
print(f"  Base: {CYCLONE_OUTPUT_BASE}")
print("=" * 70)

grand_total = 0

for cyclone_name in CYCLONES:
    print(f"\n  🌀 {cyclone_name}")
    for period_label in ["before", "cyclone", "after"]:
        folder = os.path.join(CYCLONE_OUTPUT_BASE, cyclone_name, period_label)
        if not os.path.exists(folder):
            print(f"    [{period_label:8s}]  ⚠  folder not found")
            continue

        count = sum(
            len([f for f in files if f.lower().endswith('.nc')])
            for _, _, files in os.walk(folder)
        )
        grand_total += count
        print(f"    [{period_label:8s}]  {count:>4d} .nc files  →  {folder}")

print(f"\n{'='*70}")
print(f"  Grand total .nc files across all cyclones: {grand_total}")
print("=" * 70)

  VERIFICATION — Cyclone output folder
  Base: Y:\level 3 ocean basic files\Basic_filters_cyclones

  🌀 Shakti
    [before  ]    23 .nc files  →  Y:\level 3 ocean basic files\Basic_filters_cyclones\Shakti\before
    [cyclone ]     8 .nc files  →  Y:\level 3 ocean basic files\Basic_filters_cyclones\Shakti\cyclone
    [after   ]    28 .nc files  →  Y:\level 3 ocean basic files\Basic_filters_cyclones\Shakti\after

  🌀 Remal
    [before  ]    24 .nc files  →  Y:\level 3 ocean basic files\Basic_filters_cyclones\Remal\before
    [cyclone ]     9 .nc files  →  Y:\level 3 ocean basic files\Basic_filters_cyclones\Remal\cyclone
    [after   ]    27 .nc files  →  Y:\level 3 ocean basic files\Basic_filters_cyclones\Remal\after

  🌀 Dana
    [before  ]    28 .nc files  →  Y:\level 3 ocean basic files\Basic_filters_cyclones\Dana\before
    [cyclone ]    10 .nc files  →  Y:\level 3 ocean basic files\Basic_filters_cyclones\Dana\cyclone
    [after   ]    27 .nc files  →  Y:\level 3 ocean basic files\Ba